# Практическое занятие 2.1

Пользуясь алгоритмом Кросс-Энтропии
обучить агента решать задачу Taxi-v3 из Gym.
Исследовать гиперпараметры алгоритма и выбрать лучшие.

# Установка зависимостей

In [2]:
%%capture

# Роботоспособность сред
!pip install swig
!pip install box2d-py
!pip install wandb

# Для рендеринга
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install -U colabgymrender
!pip install imageio==2.4.1
!pip install --upgrade AutoROM
!AutoROM --accept-license
!pip install gym[atari,accept-rom-license]

# Библиотеки

In [5]:
from typing import Optional, Any, Literal
from warnings import filterwarnings
import random
import time
import os

import torch.nn as nn
import torch.nn.functional as F
import torch

import numpy as np
import matplotlib.pyplot as plt
import wandb

import gym
from colabgymrender.recorder import Recorder

  from scipy.ndimage.filters import sobel



# Агент

In [6]:
class CrossEntropyAgent(nn.Module):
    def __init__(self,
                 state_dim: int,
                 action_n: int,
                 depth: list[int, ...],
                 dropout: float = None,
                 batch_norm: bool = False,
                 device: Literal["cpu", "cuda"] = "cuda",
                 epsilan: float = 0,
                 gamma: float = 0
                 ):
        super().__init__()
        self.__iterations = 0
        self.epsilan = epsilan
        self.gamma = gamma
        self.state_dim = state_dim
        self.dropout = dropout
        self.batch_norm = batch_norm
        self.action_n = action_n
        self.depth = depth
        self.layers = [state_dim, *depth, action_n]
        self.head_block_index = len(self.layers) - 2
        self.device = torch.device(device)

        self.deep_q_network = nn.Sequential(*[
            self.__get_block(index, in_features, out_features)
            for index, (in_features, out_features) in enumerate(zip(self.layers, self.layers[1:]))
        ])

        self.optimizer = torch.optim.Adam(self.parameters(), lr=15e-3)
        self.criterion = nn.CrossEntropyLoss()

        self.to_ndarray = lambda x: x.detach().cpu().numpy()
        self.to_cuda_tenzor = lambda x: torch.FloatTensor(x).to(self.device)
        self.normalize = lambda c: c / np.sum(c)

    def probs_smoothing(self, probs: np.ndarray) -> np.ndarray:
      return self.normalize((self.epsilan * self.__uniform_policy) + ((1.0 - self.epsilan) * probs))

    def exponential_epsilan_sheduler_step(self) -> None:
      self.__iterations += 1
      self.epsilan /= np.exp(self.__iterations * self.gamma)

    def __get_block(self,
                    index: int,
                    in_features: int,
                    out_features: int
                    ) -> nn.Sequential:

        _block = nn.Sequential(nn.Linear(in_features, out_features))
        if index != self.head_block_index:
            _block.append(nn.ReLU())
            if self.batch_norm:
                _block.append(nn.BatchNorm1d(out_features))
            if self.dropout:
                _block.append(nn.Dropout(self.dropout))
        return _block.to(self.device)

    def forward(self,
                state: torch.Tensor | np.ndarray | list,
                from_logites: bool = True
                ) -> torch.FloatTensor:

        if not isinstance(state, torch.Tensor):
          state = self.to_cuda_tenzor(state)
        output = self.deep_q_network(state)  # Логиты

        if not from_logites:
          output = F.softmax(output)  # Распределение вероятностей
        return output

    def get_action(self, state: torch.Tensor | np.ndarray | list) -> int:
        """
        Возвращает действие агента A при условии состояния S
        :param state: текущее состояние
        :return: P(A | S)
        """

        self.deep_q_network.eval()
        with torch.no_grad():
          probs = self.forward(state, from_logites=False)

        probs = self.to_ndarray(probs)
        probs = self.probs_smoothing(probs)
        action = np.random.choice(self.action_n, p=probs)

        self.deep_q_network.train()
        return action

    def step(self, elite_trajectories: list) -> float:
        elite_states = []
        elite_actions = []

        for trajectory in elite_trajectories:
            elite_states.extend(trajectory["states"])
            elite_actions.extend(trajectory["actions"])

        elite_states = self.to_cuda_tenzor(elite_states)
        elite_actions = torch.LongTensor(elite_actions).to(self.device)

        preds = self.forward(elite_states)
        loss = self.criterion(preds, elite_actions)

        loss.backward()
        self.optimizer.step()
        self.exponential_epsilan_sheduler_step()
        self.optimizer.zero_grad()
        return loss.item()

  and should_run_async(code)



# Среда

In [7]:
class SandBoxEnvironment:
    def __init__(self,
                 stage: Literal["train", "test"],
                 agent: CrossEntropyAgent,
                 env_name: str,
                 render_mode: Optional[str] = "human",
                 **env_kwargs
                 ):
        env = gym.make(env_name, **env_kwargs)
        self.env = env if stage == "train" else Recorder(env, f"./{stage}-videos")
        self.stage = stage
        self.agent = agent
        self.render_mode = render_mode

        self.current_iteration = 0
        self.save_top_k = None
        self.save_by = None
        self.iterations = None
        self.start_time = None
        self.mode = None

        self.models = []
        self.rewards = {
            "elite": [],
            "all": [],
            "loss": [],
            "epsilan": [],
            "q_param": [],
        }

        self.total_rewards = lambda trajectories: [trajectory["total_rewards"] for trajectory in trajectories]
        self.iter_mean_rewards = lambda trajectories: np.round(np.mean(self.total_rewards(trajectories)), 2)
        self.time = lambda: round(time.time() - self.start_time, 2) if self.start_time else -1

    def define_checkpoint(self) -> dict:
      best_finder = np.argmax if self.mode == "max" else np.argmin
      best_state_index: int = best_finder([
          model[self.save_by]
          for model in self.models
          ])

      metrics = [
          f"{key}={value}"
          for key, value in self.models[best_state_index].items()
          if key != "state_dict"
          ]

      filename = "_".join(metrics) + ".pth"
      state_dict = self.models[best_state_index]["state_dict"]
      path = os.path.join(os.getcwd(), "models", filename)
      return {
          "filepath": path,
          "state_dict": state_dict
      }


    def model_checkpoint(self) -> None:
        save_each = self.iterations // self.save_top_k
        if self.current_iteration % save_each == 0:
          checkpoint_dict = self.define_checkpoint()
          torch.save(checkpoint_dict["state_dict"], checkpoint_dict["filepath"])
          self.models.clear()

    def __get_trajectory(self, trajectory_len: int = 500) -> dict:
        state = self.env.reset()  # Инициализация среды и получения начального состояния агента
        trajectory = {
            "states": [],
            "actions": [],
            "total_rewards": 0
        }

        for _ in range(trajectory_len):
            trajectory["states"].append(state)
            action = self.agent.get_action(state)  # Агент совершает действие P(A | S)
            trajectory["actions"].append(action)
            state, reward, terminated, truncated = self.env.step(action)  # Среда обновляется
            trajectory["total_rewards"] += reward
            if terminated:
              break
        return trajectory

    def get_trajectories(self, trajectory_len: int, trajectory_n: int) -> list:
        return [
            self.__get_trajectory(trajectory_len)
            for _ in range(trajectory_n)
        ]

    def get_elite_trajectories(self, trajectories: list, q_param: Optional[float] = 0.85) -> list:
        total_rewards = self.total_rewards(trajectories)
        quantile = np.quantile(total_rewards, q_param)
        return [
            trajectory
            for trajectory in trajectories
            if trajectory["total_rewards"] > quantile
        ]

    def on_epoch_end(self, trajectories: list, elite_trajectories: list) -> None:
        mean_reward = self.iter_mean_rewards(trajectories)
        elite_mean_reward = self.iter_mean_rewards(elite_trajectories)
        time = self.time()

        self.rewards["all"].append(mean_reward)
        self.rewards["elite"].append(elite_mean_reward)

        loss = self.rewards["loss"][~0]
        q_param = self.rewards["q_param"][~0]
        epsilan = self.rewards["epsilan"][~0]

        print(f"| Time: {time} sec")
        print(f"\n| Iteration {self.current_iteration} of {self.iterations} |")
        print(f"| Mean Reward: {mean_reward} | Elite Mean Reward: {elite_mean_reward} |")
        print(f"| Loss: {round(loss, 3)} | Quantile: {q_param} | Epsilan: {round(epsilan, 3)} |\n")

        if len(self.rewards["q_param"]) > 0:
          wandb.log({key: value[~0] for key, value in self.rewards.items()})

        if self.save_top_k:
          self.models.append({
              "state_dict": self.agent.state_dict(),
              "loss": round(loss, 3),
              "iter": self.current_iteration,
              "mean_reward": mean_reward
              })
          self.model_checkpoint()

    def train(self,
              q_param: Optional[float] = 0.65,
              iterations: Optional[int] = 100,
              trajectory_len: Optional[int] = 500,
              trajectory_n: Optional[int] = 32,
              save_top_k: int = None,
              save_by: Literal["loss", "mean_reward"] = "loss",
              mode: Literal["min", "max"] = "min"
              ):
        print(f"\n| Training initialized  🚀|\n{time.asctime()}\n")
        self.start_time = time.time()
        self.iterations = iterations
        self.save_top_k = save_top_k
        self.save_by = save_by
        self.mode = mode

        for iteration in range(iterations):
            self.rewards["q_param"].append(q_param)
            self.current_iteration = iteration
            trajectories = self.get_trajectories(trajectory_len, trajectory_n)
            elite_trajectories = self.get_elite_trajectories(trajectories, q_param)
            if self.stage == "train":
              loss = self.agent.step(elite_trajectories)
              self.rewards["loss"].append(loss)
              self.rewards["epsilan"].append(self.agent.epsilan)
              self.on_epoch_end(trajectories, elite_trajectories)

        self.env.close()
        total_time = self.time()
        print(f"\n| Training finished  🚀|\nTotal time: {total_time} sec \ {round(total_time / 60, 3)} min\n")

# Стартовые функции

In [8]:
def seed_everything(seed: Optional[int] = 42) -> int:
    random.seed(seed)
    np.random.seed(seed)
    return seed


def setup_model(model: nn.Module, stage: str) -> nn.Module:
  stage_setup = True if stage == "train" else False
  for param in model.parameters():
    param.requires_grad = stage_setup
  model.train(stage_setup)
  return model


def check_sources():
    path = os.path.join(os.getcwd(), "models")
    if not os.path.exists(path):
        os.mkdir(path)


def on_startup():
  filterwarnings("ignore")  # Игнорировать предупреждения
  seed_everything(42)  # Фиксация датчика случайных чисел
  check_sources()

# Запуск обучения

In [ ]:
# Создаём агента
agent = CrossEntropyAgent(
    state_dim=8,
    depth=[1024, 512, 64],
    action_n=4,
    dropout=None,
    batch_norm=False,
    device="cuda" if torch.cuda.is_available() else "cpu",
    epsilan=0,
    gamma=1,
)

# 1000 траекторий, Макс длина 10000, 150 эпизодов, gamma q=0.2 lr 0.001
state_dict =  torch.load("/content/loss=0.721_iter=125_mean_reward=148.88.pth")
agent.load_state_dict(state_dict)

# Создаём среду
env = SandBoxEnvironment(
    stage="train",
    agent=agent,  # Агент
    env_name="LunarLander-v2",  # Имя среды (v2)
)


if __name__ == "__main__":
  on_startup()

  wandb.init(
      project="deep-reinforcement-learning",
      name="exp-eps-sheduler | TL",
      tags=["iterations 200", "SGD", "lr 15e-4", "depth 1024x256x64", "trajectory_n 1000", "trajectory_len 5_000", "Q 0.6", "gamma 95e-5"],
      job_type="train"
      )

  # Запуск обучения
  env.train(
      iterations=200,
      trajectory_n=500,
      trajectory_len=1500,
      q_param=0.9,
      save_top_k=40,
      save_by="mean_reward",
      mode="max"
  )

all,▁
elite,▁
epsilan,▁
loss,▁
q_param,▁
all,135.68
elite,178.32
epsilan,0.0
loss,0.76017
q_param,0.2



| Training initialized  🚀|
Mon Oct 16 20:21:26 2023


# Запуск тестирования

In [ ]:
# Создаём агента
agent = CrossEntropyAgent(
    state_dim=8,
    depth=[1024, 512, 64],
    action_n=4,
    device="cuda" if torch.cuda.is_available() else "cpu",
)


state_dict =  torch.load("/content/loss=0.721_iter=125_mean_reward=148.88.pth")
agent.load_state_dict(state_dict)


# Создаём среду
env = SandBoxEnvironment(
    stage="test",
    agent=agent,  # Агент
    env_name="LunarLander-v2",  # Имя среды (v2)
)


if __name__ == "__main__":
  on_startup()

  wandb.init(
      project="deep-reinforcement-learning",
      name="test runs",
      tags=["test", "rendering"],
      job_type="test"
      )

  # Запуск обучения
  env.train(
      iterations=1,  # Кол-во итераций 50
      trajectory_n=20,  # Кол-во траекторий 500
      trajectory_len=10_000,  # Длина траекторий 170
      save_top_k=None
  )

  deprecation(

  deprecation(

See here for more information: https://www.gymlibrary.ml/content/api/
  deprecation(



<IPython.core.display.Javascript object>

wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc



| Training initialized  🚀|
Mon Oct 16 07:09:35 2023


| Training finished  🚀|
Total time: 86.29 sec \ 1.438 min
